# Toy CLIP from scratch (NumPy)
Contrastive image-text learning on synthetic shapes.

## 1. Setup and data
We render synthetic 8x8 shape images and pair each one with a caption built from templates and synonyms.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))
import numpy as np
from data import make_dataset, full_vocab, bow, SHAPES, COLORS, SYN, HELD_OUT
from clip import init_encoder, encode, train_clip, info_nce, recall_at_k
rng = np.random.default_rng(42)
tr = make_dataset(3000, rng, 'train'); te = make_dataset(200, rng, 'test'); ho = make_dataset(200, rng, 'heldout')
V = full_vocab()
print(len(V), 'words;', tr['captions'][:3], tr['labels'][:3])

## 2. Look at one image as characters
Each pixel prints as R/G/B for its strongest channel, Y for yellow (red and green both high), or '.' for background.

In [ ]:
img = tr['images'][0]
for row in img:
    print(''.join('.' if px.max() < 0.4 else 'Y' if min(px[0], px[1]) > 0.5 else 'RGB'[int(px.argmax())] for px in row))
print(tr['labels'][0], '|', tr['captions'][0])

## 3. Gradient check of InfoNCE and the L2-norm backward pass

In [ ]:
from clip import encoder_backward
p = {**init_encoder(192, 16, 8, rng, 'img_'), **init_encoder(len(V), 16, 8, rng, 'txt_')}
Xi = tr['images'][:6].reshape(6, -1); Xt = bow(tr['captions'][:6], V)
def L(p):
    ei,_ = encode(p, Xi, 'img_'); et,_ = encode(p, Xt, 'txt_'); return info_nce(ei, et, 0.1)[0]
ei, ci = encode(p, Xi, 'img_'); et, ct = encode(p, Xt, 'txt_')
_, dei, det = info_nce(ei, et, 0.1)
g = encoder_backward(p, ci, dei, 'img_')
k, idx, eps = 'img_W1', (3, 2), 1e-5
p[k][idx] += eps; lp = L(p); p[k][idx] -= 2*eps; lm = L(p); p[k][idx] += eps
print('analytic', g[k][idx], 'numeric', (lp - lm) / (2*eps))

## 4. Train the toy CLIP

In [ ]:
p = {**init_encoder(192, 128, 32, rng, 'img_'), **init_encoder(len(V), 64, 32, rng, 'txt_')}
hist = train_clip(p, tr['images'].reshape(3000, -1), bow(tr['captions'], V), epochs=40, batch=64, lr=3e-3, tau=0.1, rng=rng)
print('loss', hist[0], '->', hist[-1], ' chance =', np.log(64))

## 5. Retrieval recall@k

In [ ]:
ei,_ = encode(p, te['images'].reshape(200, -1), 'img_'); et,_ = encode(p, bow(te['captions'], V), 'txt_')
S = ei @ et.T
conc = np.array([[a == b for b in te['labels']] for a in te['labels']])
for k in (1, 5):
    print(f'R@{k}  i2t concept {recall_at_k(S, k, conc):.3f}  t2i concept {recall_at_k(S.T, k, conc.T):.3f}  i2t instance {recall_at_k(S, k, np.eye(200, dtype=bool)):.3f}')

## 6. Zero-shot classification with prompts, including held-out colour-shape combos

In [ ]:
def zs(d):
    e,_ = encode(p, d['images'].reshape(len(d['images']), -1), 'img_')
    protos = []
    for s in SHAPES:
        t,_ = encode(p, bow([f'a photo of a {w}' for w in SYN[s]], V), 'txt_'); m = t.mean(0); protos.append(m / np.linalg.norm(m))
    y = np.array([SHAPES.index(l[2]) for l in d['labels']])
    return ((e @ np.stack(protos).T).argmax(1) == y).mean()
print('shape zero-shot  test', zs(te), ' held-out combos', zs(ho), ' chance 0.25')

## 7. Full smoke run
Run `python run_smoke.py` from the repo root. It writes `results/`.